In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
from torchvision.transforms.functional import to_tensor

sys.path.append("..")
from src import *

# Compute Metrics

In [ ]:
dataset = ObjaverseDataset3D()

In [ ]:
TESTSET_DIR = Path("dataset/test")
REND_DIR = Path("renderings")
TEST_DIR = Path("tests")
dirs = [
    "gt",
    "sd15_mlsd",
    "sd15_ours",
    "sdxl_ours",
    "sdxl_mlsd_llite",
    "sdxl_ours_llite",
]
testset = pd.read_json(TESTSET_DIR / "metadata.jsonl", orient="records", lines=True)
testset.index = pd.Series(testset.uv_file_name.map(lambda x: Path(x).stem), name="uid")

In [ ]:
metrics: dict[str, Metric] = {
    "psnr": PSNRMetric(),
    "ssim": SSIMMetric(),
    "lpips": LPIPSMetric(),
    "fid": FIDMetric(),
    "clipiqa": CLIPIQAMetric(),
    "brisque": BRISQUEMetric(),
}

In [ ]:
def path2tensor(path: Path) -> torch.Tensor:
    with Image.open(path) as img:
        return to_tensor(img.convert("RGB").resize((512, 512))).unsqueeze(0)

In [ ]:
y_tag = "sdxl_ours_llite"
view=0

uids = testset.index
y_tex = torch.empty((len(uids), 3, 512, 512))
gt_tex = torch.empty_like(y_tex)
y_ren = torch.empty_like(y_tex)
gt_ren = torch.empty_like(y_tex)
captions=[]

for i, uid in tqdm(enumerate(uids)):
    y_tex[i] = path2tensor(TEST_DIR / y_tag / f"{uid}.png")
    gt_tex[i] = path2tensor(TESTSET_DIR / "diffuse" / f"{uid}.png")
    y_ren[i] = path2tensor(REND_DIR / y_tag / f"{uid[:-2]}_0.png")
    gt_ren[i] = path2tensor(REND_DIR / "gt" / f"{uid[:-2]}_0.png")
    captions.append(testset.loc[uid].caption)

In [ ]:
for k, metric in metrics.items():
    if metric.need_renders:
        m = metric(y_ren, gt_ren)
    else:
        m = metric(y_tex, gt_tex)
    cprint(f"green:{k}", f"blue:{m:.3f}")

### Stable Diffusion 1.5
| Model             |  $\text{PSNR} ↑$ |  $\text{SSIM} ↑$ | $\text{LPIPS} ↓$ | $\text{FID} ↓$ | $\text{CLIP-IQA} ↑$ | $\text{BRISQUE} ↓$ |
| ----------------- | :----------------: | :----------------: | :----------------: | :--------------: | :-------------------: | :------------------: |
| `sd15_mlsd`       |      $8.139$      |      $0.243$      |      $0.811$      |    $233.35$    |        $0.797$       |       $\mathbf{44.49}$      |
| `sd15_ours`       |      $\mathbf{8.553}$      |      $\mathbf{0.279}$      |      $\mathbf{0.789}$      |    $\mathbf{229.43}$    |        $\mathbf{0.805}$       |       $51.87$      |

### Stable Diffusion XL

| Model             |  $\text{PSNR} ↑$ |  $\text{SSIM} ↑$ | $\text{LPIPS} ↓$ | $\text{FID} ↓$ | $\text{CLIP-IQA} ↑$ | $\text{BRISQUE} ↓$ |
| ----------------- | :----------------: | :----------------: | :----------------: | :--------------: | :-------------------: | :------------------: |
| `sdxl_ours`       |      $8.600$     |      $0.314$     |      $0.802$     |    $\mathbf{215.75}$    |       $0.795$       |      $50.505$      |
| `sdxl_mlsd_llite` |      $8.226$     |      $0.125$     |      $0.907$     |    $300.63$   |       $0.792$       |      $\mathbf{35.111}$     |
| `sdxl_ours_llite` | $\mathbf{9.105}$ | $\mathbf{0.367}$ | $\mathbf{0.783}$ |    $260.55$   |       $\mathbf{0.803}$       |      $50.728$      |
